# 07 - Árvore de decisão e explicabilidade

Este notebook fecha a trilha com uma pergunta executiva: **ganhamos explicabilidade sem comprometer demais a performance?**

A árvore de decisão é avaliada como modelo interpretável e comparada com os candidatos já validados.

Roteiro da etapa:
- medir desempenho da árvore com as mesmas métricas do projeto;
- comparar com modelos de referência;
- verificar se a recomendação principal muda ou se a árvore entra como apoio de comunicação e governança.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

BASE_DIR = Path("..")
DATA_PATH = BASE_DIR / "data" / "WineQT.csv"

wine = pd.read_csv(DATA_PATH)
wine["target"] = (wine["quality"] >= 7).astype(int)

feature_cols = [c for c in wine.columns if c not in {"quality", "target", "Id"}]
X = wine[feature_cols]
y = wine["target"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

models = {
    "decision_tree": DecisionTreeClassifier(
        max_depth=4, random_state=42, class_weight="balanced"
    ),
    "logreg_balanced": LogisticRegression(
        max_iter=4000, class_weight="balanced", random_state=42
    ),
    "random_forest_balanced": RandomForestClassifier(
        n_estimators=300, random_state=42, class_weight="balanced_subsample"
    ),
    "histgradient_boosting": HistGradientBoostingClassifier(random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    proba = (
        model.predict_proba(X_val)[:, 1]
        if hasattr(model, "predict_proba")
        else model.decision_function(X_val)
    )
    results[name] = {
        "accuracy": round(accuracy_score(y_val, pred), 4),
        "precision": round(precision_score(y_val, pred, zero_division=0), 4),
        "recall": round(recall_score(y_val, pred, zero_division=0), 4),
        "f1": round(f1_score(y_val, pred, zero_division=0), 4),
        "roc_auc": round(roc_auc_score(y_val, proba), 4),
    }

print(json.dumps(results, indent=2, ensure_ascii=False))

results_df = pd.DataFrame.from_dict(results, orient="index").reset_index()
results_df = results_df.rename(columns={"index": "modelo"})

plt.figure(figsize=(10, 5))
sns.barplot(
    data=results_df, x="modelo", y="f1", hue="modelo", palette="viridis", legend=False
)
plt.title("Comparação de F1 entre modelos na validação")
plt.xlabel("Modelo")
plt.ylabel("F1-score")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

model_tree = DecisionTreeClassifier(
    max_depth=4, random_state=42, class_weight="balanced"
)
model_tree.fit(X_train, y_train)
feature_importance = (
    pd.Series(model_tree.feature_importances_, index=feature_cols)
    .sort_values(ascending=False)
    .head(10)
)

plt.figure(figsize=(8, 5))
feature_importance.sort_values().plot(kind="barh", color="darkorange")
plt.title("Top 10 variáveis mais importantes na árvore de decisão")
plt.xlabel("Importância")
plt.ylabel("Variável")
plt.tight_layout()
plt.show()

## Decisão de negócio

A árvore de decisão cumpre muito bem o papel de explicabilidade: facilita comunicação com áreas não técnicas e torna regras mais transparentes.

Por outro lado, quando o critério principal é desempenho robusto em problema desbalanceado, a decisão final deve permanecer no modelo com melhor equilíbrio de **F1**, **recall** e métricas probabilísticas.

Conclusão narrativa: a árvore entra como instrumento de interpretação e governança, enquanto a recomendação principal continua ancorada na melhor performance global validada.